In [66]:
import re
from pathlib import Path

import matplotlib.cm as cm
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import seaborn as sns

from analyses.plotting.scatter_with_regression import \
    plot_scatter_with_regression
from analyses.utils.aggregate_data import aggregate_data
from processing.avo import (correct_pnc_using_dynamic_cutoff,
                            correlate_pnc_pm10,
                            extract_detailed_data_zip_to_polars_dfs)


def _wavelength_to_color(wavelengths):
    norm = mcolors.Normalize(vmin=min(wavelengths), vmax=max(wavelengths))
    colormap = cm.get_cmap("rainbow")
    return [mcolors.to_hex(colormap(norm(w))) for w in wavelengths]

def plot_time_series(
    df: pl.DataFrame,
    channels_and_colors: dict,
    highlight_channel: str = None,
    dtm: str = "dtm",
    output_path: Path = Path("ae31_plot.png"),
    figsize: tuple = (10, 5),
    title: str = "Aethalometer AE31 hourly data (Dagoretti Corner, NRB)",
    xlabel: str = "Date",
    ylabel: str | tuple[str] | None = "ylabel",
    legend_title: str = "Channels",
    legend_loc: str = "upper right",
    legend_fontsize: int = 8,
    plot_type: str = "line",  # Options: "line" or "scatter"
    which_y_axes: tuple[int] | None = None,  # 1 = left, 2 = right
    grid: bool = False,  # Enable or disable grid
):
    assert plot_type in {"line", "scatter"}, "plot_type must be 'line' or 'scatter'"

    df_pd = df.to_pandas()
    x = df_pd[dtm]

    fig, ax1 = plt.subplots(figsize=figsize)
    ax2 = ax1.twinx()

    # Set y-axis labels
    if isinstance(ylabel, list) and len(ylabel) == 2:
        ax1.set_ylabel(ylabel[0])
        ax2.set_ylabel(ylabel[1])
    elif isinstance(ylabel, str):
        ax1.set_ylabel(ylabel)
    elif ylabel is not None:
        raise ValueError("ylabel must be a string, a list of two strings, or None")

    # Assign all channels to axis 1 by default
    if which_y_axes is None:
        which_y_axes = [1] * len(channels_and_colors)

    if len(which_y_axes) != len(channels_and_colors):
        raise ValueError("Length of which_y_axes must match number of channels.")

    # Map channels to their axis
    ch_axis = dict(zip(channels_and_colors.keys(), which_y_axes))

    for ch, color in channels_and_colors.items():
        y = df_pd[ch]
        ax = ax1 if ch_axis[ch] == 1 else ax2

        if ch == highlight_channel:
            ax.scatter(x, y, color=color, label=ch, s=10, zorder=5)
        else:
            if plot_type == "scatter":
                ax.scatter(x, y, color=color, label=ch, s=8)
            else:
                ax.plot(x, y, color=color, label=ch, linewidth=1.0)

    ax1.set_title(title)
    ax1.set_xlabel(xlabel)

    # Grid control
    ax1.grid(grid)
    ax2.grid(False)

    # Enable horizontal (x-axis) tick marks
    ax1.tick_params(axis='x', which='both', direction='out', bottom=True, top=False, length=4)
    ax2.tick_params(axis='x', which='both', direction='out', bottom=True, top=False, length=4)

    # Merge and display legend
    handles1, labels1 = ax1.get_legend_handles_labels()
    handles2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(
        handles1 + handles2,
        labels1 + labels2,
        title=legend_title,
        loc=legend_loc,
        fontsize=legend_fontsize,
        title_fontsize=legend_fontsize,
    )

    fig.autofmt_xdate()
    fig.tight_layout()
    plt.savefig(output_path, dpi=300)
    plt.close()

    print(f"✅ Time series {plot_type} plot saved to {output_path}")    

def plot_diurnal_by_wavelength(
    df: pl.DataFrame,
    channels_and_colors: dict,
    dtm: str = "dtm",
    plot_type: str = "box",  # Options: "box", "violin", "swarm"
    output_path: Path = Path("ae31_diurnal.png"),
    figsize: tuple = (10, 5),
    title: str = "Diurnal variability of aerosol absorption (Dagoretti Corner, NRB)",
    xlabel: str = "Hour of Day [UTC]",
    ylabel: str = "Aerosol particle absorption (ng/m³)",
    legend_title: str = "Channels",
    legend_loc: str = "upper right",
    legend_fontsize: int = 8,
):
    assert plot_type in {"box", "violin", "swarm"}, "plot_type must be 'box', 'violin', or 'swarm'"

    # 1. Add hour column
    df = df.with_columns(
        pl.col(dtm).dt.hour().alias("hour")
    )

    # 2. Melt to long format
    melted = df.melt(
        id_vars=["hour"],
        value_vars=list(channels_and_colors.keys()),
        variable_name="channel",
        value_name="value"
    )

    # 3. Filter negative values
    melted = melted.filter(pl.col("value") >= 0)

    # 4. Convert to pandas for Seaborn
    df_plot = melted.to_pandas()

    # 5. Ensure channel order by wavelength
    channel_order = list(channels_and_colors.keys())

    # 6. Plot
    plt.figure(figsize=figsize)
    ax = plt.gca()

    if plot_type == "box":
        sns.boxplot(
            data=df_plot,
            x="hour", y="value", hue="channel",
            hue_order=channel_order,
            palette=channels_and_colors,
            width=0.7,
            fliersize=2,
            ax=ax,
        )
    elif plot_type == "violin":
        sns.violinplot(
            data=df_plot,
            x="hour", y="value", hue="channel",
            hue_order=channel_order,
            palette=channels_and_colors,
            linewidth=0.8,
            ax=ax,
        )
    elif plot_type == "swarm":
        sns.swarmplot(
            data=df_plot,
            x="hour", y="value", hue="channel",
            hue_order=channel_order,
            palette=channels_and_colors,
            size=2,
            ax=ax,
        )

    # Final touches
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)

    # Legend
    ax.legend(
        title=legend_title,
        loc=legend_loc,
        fontsize=legend_fontsize,
        title_fontsize=legend_fontsize,
    )

    plt.tight_layout()
    plt.savefig(output_path, dpi=300)
    plt.close()

    print(f"✅ Diurnal {plot_type} plot saved to {output_path}")


import polars as pl
import numpy as np

# def combine_avo_fidas_pm_dataframes(df_avo: pl.DataFrame, df_fidas: pl.DataFrame) -> pl.DataFrame:
#     """
#     Aligns and merges AVO and FIDAS data for PM1, PM2.5, and PM10 based on common timestamps.
#     Converts FIDAS from mg/m³ to µg/m³.
#     Adds '_avo [µg/m3]' and '_fidas [µg/m3]' suffixes to PM columns.

#     Args:
#         df_avo (pl.DataFrame): AVO dataframe with 'dtm' and PMx (ug/cm3) columns.
#         df_fidas (pl.DataFrame): FIDAS dataframe with 'dtm' and PMx [mg/m³] columns.

#     Returns:
#         pl.DataFrame: Joined dataframe with renamed columns and aligned timestamps.
#     """
#     pm_vars = ["PM1", "PM2.5", "PM10"]
#     unit_label = " [µg/m3]"

#     # Ensure naive timestamps
#     df_avo = df_avo.with_columns(pl.col("dtm").dt.replace_time_zone(None))
#     df_fidas = df_fidas.with_columns(pl.col("dtm").dt.replace_time_zone(None))

#     # AVO columns: select and rename
#     avo_cols = ["dtm"]
#     avo_renames = {}
#     for pm in pm_vars:
#         src_col = f"{pm} [ug/m3]"
#         dst_col = f"{pm}_avo{unit_label}"
#         if src_col in df_avo.columns:
#             avo_cols.append(src_col)
#             avo_renames[src_col] = dst_col

#     df_avo = df_avo.select(avo_cols).rename(avo_renames)

#     # FIDAS columns: convert and rename
#     fidas_cols = ["dtm"]
#     fidas_renames = {}
#     for pm in pm_vars:
#         src_col = f"{pm} [mg/m³]"
#         dst_col = f"{pm}_fidas{unit_label}"
#         if src_col in df_fidas.columns:
#             fidas_cols.append(src_col)
#             fidas_renames[src_col] = dst_col

#     df_fidas = df_fidas.select(fidas_cols).with_columns([
#         (pl.col(src) * 1000).alias(dst) for src, dst in fidas_renames.items()
#     ]).select(["dtm"] + list(fidas_renames.values()))

#     # Align on common timestamps
#     dt_common = pl.Series("dtm", np.intersect1d(
#         df_avo["dtm"].to_numpy(),
#         df_fidas["dtm"].to_numpy()
#     ))

#     df_avo = df_avo.filter(pl.col("dtm").is_in(dt_common)).sort("dtm")
#     df_fidas = df_fidas.filter(pl.col("dtm").is_in(dt_common)).sort("dtm")

#     # Join
#     df_joined = df_avo.join(df_fidas, on="dtm")

#     return df_joined
def combine_avo_fidas_pm_dataframes(df_avo: pl.DataFrame, df_fidas: pl.DataFrame) -> pl.DataFrame:
    """
    Aligns and merges AVO and FIDAS data for PM1, PM2.5, PM10, and PNC based on common timestamps.
    Converts FIDAS PM values from mg/m³ to µg/m³.
    Adds '_avo [unit]' and '_fidas [unit]' suffixes to columns.

    Args:
        df_avo (pl.DataFrame): AVO dataframe with 'dtm' and PMx [ug/m3] and PNC [1/cm³].
        df_fidas (pl.DataFrame): FIDAS dataframe with 'dtm', PMx [mg/m³], and PNC [1/cm³].

    Returns:
        pl.DataFrame: Joined dataframe with renamed columns and aligned timestamps.
    """
    pm_vars = ["PM1", "PM2.5", "PM10"]
    pnc_var = "PNC"
    unit_pm = " [µg/m3]"
    unit_pnc = " [1/cm³]"

    # Ensure naive timestamps
    df_avo = df_avo.with_columns(pl.col("dtm").dt.replace_time_zone(None))
    df_fidas = df_fidas.with_columns(pl.col("dtm").dt.replace_time_zone(None))

    # --- AVO columns: PM + PNC ---
    avo_cols = ["dtm"]
    avo_renames = {}

    for pm in pm_vars:
        src_col = f"{pm} [ug/m3]"
        dst_col = f"{pm}_avo{unit_pm}"
        if src_col in df_avo.columns:
            avo_cols.append(src_col)
            avo_renames[src_col] = dst_col

    src_col = f"{pnc_var} [1/cm3]"
    dst_col = f"{pnc_var}_avo{unit_pnc}"
    if src_col in df_avo.columns:
        avo_cols.append(src_col)
        avo_renames[src_col] = dst_col

    df_avo = df_avo.select(avo_cols).rename(avo_renames)

    # --- FIDAS columns: PM (convert) + PNC ---
    fidas_cols = ["dtm"]
    fidas_renames = {}
    conversions = []

    for pm in pm_vars:
        src_col = f"{pm} [mg/m³]"
        dst_col = f"{pm}_fidas{unit_pm}"
        if src_col in df_fidas.columns:
            fidas_cols.append(src_col)
            fidas_renames[src_col] = dst_col
            conversions.append((src_col, dst_col))  # needs ×1000

    src_col = f"{pnc_var} [1/cm³]"
    dst_col = f"{pnc_var}_fidas{unit_pnc}"
    if src_col in df_fidas.columns:
        fidas_cols.append(src_col)
        fidas_renames[src_col] = dst_col

    # Select only the columns we'll convert or rename
    df_fidas = df_fidas.select(fidas_cols)

    # Apply conversion and renaming in one step for converted columns
    converted = [(pl.col(src) * 1000).alias(dst) for src, dst in conversions]

    # Add direct renames for PNC (no conversion)
    direct_renames = [pl.col(src).alias(dst) for src, dst in fidas_renames.items() if src not in dict(conversions)]

    # Build clean dataframe
    df_fidas = df_fidas.select(["dtm"] + converted + direct_renames)

    # Align on common timestamps
    dt_common = pl.Series("dtm", np.intersect1d(
        df_avo["dtm"].to_numpy(),
        df_fidas["dtm"].to_numpy()
    ))

    df_avo = df_avo.filter(pl.col("dtm").is_in(dt_common)).sort("dtm")
    df_fidas = df_fidas.filter(pl.col("dtm").is_in(dt_common)).sort("dtm")

    return df_avo.join(df_fidas, on="dtm")

# def plot_pm_comparison_from_dataframe(df: pl.DataFrame,
#                                       output_dir: Path,
#                                       width: float = 6,
#                                       height: float = 5) -> None:
#     """
#     Generate scatter + regression plots from joined PM dataframe.

#     Args:
#         df (pl.DataFrame): Must contain 'PMx (ug/cm3)' and 'PMx (ug/cm3)_fidas' columns.
#         output_dir (Path): Directory to save plots.
#         width (float): Plot width in inches.
#         height (float): Plot height in inches.
#     """
#     output_dir.mkdir(parents=True, exist_ok=True)
#     pm_cols = ["PM1", "PM2.5", "PM10"]

#     for pm in pm_cols:
#         avo_col = f"{pm}_avo [µg/m3]"
#         fidas_col = f"{pm}_fidas [µg/m3]"

#         if avo_col not in df.columns or fidas_col not in df.columns:
#             continue

#         df_pair = df.select([avo_col, fidas_col]).drop_nulls()
#         x = df_pair[fidas_col].to_numpy()
#         y = df_pair[avo_col].to_numpy()

#         # Clean up
#         valid = np.isfinite(x) & np.isfinite(y)
#         x = x[valid]
#         y = y[valid]

#         # Safety checks
#         if len(x) < 2 or np.std(x) == 0 or np.std(y) == 0:
#             print(f"⚠️ Skipping {pm}: insufficient or invalid data (n={len(x)})")
#             continue

#         slope, intercept = np.polyfit(x, y, 1)
#         r = np.corrcoef(x, y)[0, 1]

#         plt.figure(figsize=(width, height))
#         plt.scatter(x, y, alpha=0.5, label="Data")
#         x_fit = np.linspace(x.min(), x.max(), 100)
#         y_fit = slope * x_fit + intercept
#         plt.plot(x_fit, y_fit, color="red", lw=2, label="Fit")
#         plt.xlabel(f"FIDAS {pm} [µg/m³]")
#         plt.ylabel(f"AVO {pm} [µg/m³]")
#         plt.title(f"{pm} comparison")
#         plt.text(0.05, 0.95,
#                  f"$r$ = {r:.3f}\n$y = {slope:.2f}x + {intercept:.2f}$",
#                  transform=plt.gca().transAxes,
#                  fontsize=10, va='top', ha='left',
#                  bbox=dict(facecolor='white', alpha=0.7))
#         plt.grid(True)
#         plt.legend()
#         plt.tight_layout()
#         plt.savefig(output_dir / f"nrb_avo_fidas_{pm.replace('.', '')}.png", dpi=150)
#         plt.close()
# def plot_pm_comparison_from_dataframe(df: pl.DataFrame,
#                                       output_dir: Path,
#                                       width: float = 6,
#                                       height: float = 5) -> None:
#     """
#     Generate scatter + regression plots from joined PM dataframe.
#     Adds 1:1 dashed reference line and saves plots with 'nrb_avo_fidas_' prefix.

#     Args:
#         df (pl.DataFrame): Must contain 'PMx_avo [µg/m3]' and 'PMx_fidas [µg/m3]' columns.
#         output_dir (Path): Directory to save plots.
#         width (float): Plot width in inches.
#         height (float): Plot height in inches.
#     """
#     output_dir.mkdir(parents=True, exist_ok=True)
#     pm_vars = ["PM1", "PM2.5", "PM10"]
#     unit = " [µg/m3]"

#     for pm in pm_vars:
#         avo_col = f"{pm}_avo{unit}"
#         fidas_col = f"{pm}_fidas{unit}"

#         if avo_col not in df.columns or fidas_col not in df.columns:
#             print(f"⚠️ Columns missing for {pm}")
#             continue

#         df_pair = df.select([avo_col, fidas_col]).drop_nulls()
#         x = df_pair[fidas_col].to_numpy()
#         y = df_pair[avo_col].to_numpy()

#         # Filter out invalid values
#         valid = np.isfinite(x) & np.isfinite(y)
#         x = x[valid]
#         y = y[valid]

#         if len(x) < 2 or np.std(x) == 0 or np.std(y) == 0:
#             print(f"⚠️ Skipping {pm}: insufficient or invalid data (n={len(x)})")
#             continue

#         # Regression
#         slope, intercept = np.polyfit(x, y, 1)
#         r = np.corrcoef(x, y)[0, 1]

#         # Axis limits
#         lim_min = min(x.min(), y.min())
#         lim_max = max(x.max(), y.max())
#         padding = 0.05 * (lim_max - lim_min)
#         lim_min -= padding
#         lim_max += padding

#         # Plot
#         plt.figure(figsize=(width, height))
#         plt.scatter(x, y, alpha=0.5, label="Data")
#         x_fit = np.linspace(lim_min, lim_max, 100)
#         y_fit = slope * x_fit + intercept
#         plt.plot(x_fit, y_fit, color="red", lw=2, label="Fit")
#         plt.plot(x_fit, x_fit, linestyle="--", color="gray", label="1:1 line")

#         plt.xlabel(f"FIDAS {pm} (µg/m³)")
#         plt.ylabel(f"AVO {pm} (µg/m³)")
#         plt.title(f"{pm} comparison")
#         plt.text(0.05, 0.95,
#                  f"$r$ = {r:.3f}\n$y = {slope:.2f}x + {intercept:.2f}$",
#                  transform=plt.gca().transAxes,
#                  fontsize=10, va='top', ha='left',
#                  bbox=dict(facecolor='white', alpha=0.7))
#         plt.grid(True)
#         plt.legend()
#         plt.xlim(lim_min, lim_max)
#         plt.ylim(lim_min, lim_max)
#         plt.gca().set_aspect('equal', adjustable='box')
#         plt.tight_layout()

#         plot_filename = output_dir / f"nrb_avo_fidas_{pm.replace('.', '')}.png"
#         plt.savefig(plot_filename, dpi=150)
#         plt.close()
def plot_pm_comparison_from_dataframe(df: pl.DataFrame,
                                      output_dir: Path,
                                      width: float = 6,
                                      height: float = 5) -> None:
    """
    Generate scatter + regression plots for PM1, PM2.5, PM10, and PNC from a joined dataframe.
    Adds dashed 1:1 line and uses equal axis limits.

    Args:
        df (pl.DataFrame): Must contain 'VAR_avo [unit]' and 'VAR_fidas [unit]' columns.
        output_dir (Path): Directory to save plots.
        width (float): Plot width in inches.
        height (float): Plot height in inches.
    """
    output_dir.mkdir(parents=True, exist_ok=True)

    vars_to_plot = [
        ("PM1", " [µg/m3]"),
        ("PM2.5", " [µg/m3]"),
        ("PM10", " [µg/m3]"),
        ("PNC", " [1/cm³]"),
    ]

    for var, unit in vars_to_plot:
        avo_col = f"{var}_avo{unit}"
        fidas_col = f"{var}_fidas{unit}"

        if avo_col not in df.columns or fidas_col not in df.columns:
            print(f"⚠️ Columns missing for {var}")
            continue

        df_pair = df.select([avo_col, fidas_col]).drop_nulls()
        x = df_pair[fidas_col].to_numpy()
        y = df_pair[avo_col].to_numpy()

        valid = np.isfinite(x) & np.isfinite(y)
        x = x[valid]
        y = y[valid]

        if len(x) < 2 or np.std(x) == 0 or np.std(y) == 0:
            print(f"⚠️ Skipping {var}: insufficient or invalid data (n={len(x)})")
            continue

        slope, intercept = np.polyfit(x, y, 1)
        r = np.corrcoef(x, y)[0, 1]

        lim_min = min(x.min(), y.min())
        lim_max = max(x.max(), y.max())
        padding = 0.05 * (lim_max - lim_min)
        lim_min -= padding
        lim_max += padding

        plt.figure(figsize=(width, height))
        plt.scatter(x, y, alpha=0.5, label="Data")
        x_fit = np.linspace(lim_min, lim_max, 100)
        y_fit = slope * x_fit + intercept
        plt.plot(x_fit, y_fit, color="red", lw=2, label="Fit")
        plt.plot(x_fit, x_fit, linestyle="--", color="gray", label="1:1 line")

        plt.xlabel(f"FIDAS {var}{unit}")
        plt.ylabel(f"AVO {var}{unit}")
        plt.title(f"{var} comparison")
        plt.text(0.05, 0.95,
                 f"$r$ = {r:.3f}\n$y = {slope:.2f}x + {intercept:.2f}$",
                 transform=plt.gca().transAxes,
                 fontsize=10, va='top', ha='left',
                 bbox=dict(facecolor='white', alpha=0.7))
        plt.grid(True)
        plt.legend()
        plt.xlim(lim_min, lim_max)
        plt.ylim(lim_min, lim_max)
        plt.gca().set_aspect('equal', adjustable='box')
        plt.tight_layout()

        plot_filename = output_dir / f"nrb_avo_fidas_{var.replace('.', '')}.png"
        plt.savefig(plot_filename, dpi=150)
        plt.close()

source = Path("data/level1/nrb")       # folder with .parquet files
target = Path("data/level2/nrb")      # where output should go
results = Path("results") / "aq_conference_nairobi_2025"

In [ ]:
# Read iQAir AirVisualOutdoor data downloaded manually
path = target / "avo/IQAir_Export_validated_devices_29Jun24-29Jun25_hourly.zip"
dfs = extract_detailed_data_zip_to_polars_dfs(Path(path))
print(dfs.keys())  # -> Source IDs

df_avo_nrb = dfs['x15ai15feje']
df_avo_nrb.write_parquet(target / "avo/df_avo_nrb-hourly.parquet")
df_avo_bmt = dfs['twrgmdgz53t']
df_avo_bmt.write_parquet(target / "avo/df_avo_bmt-hourly.parquet")
df_avo_mtf = dfs['wichuo199fk']
df_avo_mtf.write_parquet(target / "avo/df_avo_mtf-hourly.parquet")

In [ ]:
# plot timeseries
channels_and_colors = {
    "PNC [1/cm3]": "#1f77b4",  # Blue       — for number concentration (distinct from mass)
    "PM1 [ug/m3]": "#2ca02c",  # Green      — PM1
    "PM2.5 [ug/m3]": "#ff7f0e",  # Orange     — PM2.5
    # "PM4 [mg/m³]": "#d62728",  # Red        — PM4
    "PM10 [ug/m3]": "#9467bd",  # Purple     — PM10
    # "PMtotal [mg/m³]": "#8c564b",  # Brown      — PMtotal
}

plot_time_series(df_avo_nrb, 
                 channels_and_colors=channels_and_colors, 
                 highlight_channel='PNC [1/cm³]', 
                 plot_type="line", 
                 output_path=results / "nrb_avo_timeseries.png", 
                 figsize=(10, 5),
                 title="iQAir AirVisual Outdoor original hourly data (Dagoretti Corner, NRB)",
                 ylabel=["Aerosol particle mass concentration [µg/m3]", "Aerosol particle number concentration [1/cm3]"],
                 which_y_axes=(2, 1, 1, 1),
                 legend_loc='upper left')


In [ ]:
df_avo_nrb_corrected = correct_pnc_using_dynamic_cutoff(df_avo_nrb, pnc_february_level=5, factor=50)
correlate_pnc_pm10(df_avo_nrb_corrected)

In [ ]:
# plot timeseries
channels_and_colors = {
    "PNC [1/cm3]": "#1f77b4",  # Blue       — for number concentration (distinct from mass)
    "PM1 [ug/m3]": "#2ca02c",  # Green      — PM1
    "PM2.5 [ug/m3]": "#ff7f0e",  # Orange     — PM2.5
    # "PM4 [mg/m³]": "#d62728",  # Red        — PM4
    "PM10 [ug/m3]": "#9467bd",  # Purple     — PM10
    # "PMtotal [mg/m³]": "#8c564b",  # Brown      — PMtotal
}

plot_time_series(df_avo_nrb_corrected, 
                 channels_and_colors=channels_and_colors, 
                 highlight_channel='PNC [1/cm³]', 
                 plot_type="line", 
                 output_path=results / "nrb_avo_corrected_timeseries.png", 
                 figsize=(10, 5),
                 title="iQAir AirVisual Outdoor corrected hourly data (Dagoretti Corner, NRB)",
                 ylabel=["Aerosol particle mass concentration [µg/m3]", "Aerosol particle number concentration [1/cm3]"],
                 which_y_axes=(2, 1, 1, 1),
                 legend_loc='upper left')


In [ ]:
# Nairobi Fidas data
source = Path("data/level1/nrb")       # folder with .parquet files
target = Path("data/level2/nrb")      # where output should go
results = Path("results") / "aq_conference_nairobi_2025"
instrument_name = "fidas"
map = {'60': "PNC [1/cm³]",
        '61': "PM1 [mg/m³]",
        '62': "PM2.5 [mg/m³]",
        '63': "PM4 [mg/m³]",
        '64': "PM10 [mg/m³]",
        '65': "PMtotal [mg/m³]",
}

# aggregate data
df_fidas, path_fidas = aggregate_data(
    source=source,       # folder with .parquet files
    target=target,      # where output should go
    instrument_name=instrument_name,
    freq="hourly",
    extract_cols=map.keys(),
    statistics="mean",
)
df_fidas = df_fidas.rename({old: new for old, new in map.items() if old in df_fidas.columns})

df_fidas.write_parquet(path_fidas)

In [67]:
# Process
df_common = combine_avo_fidas_pm_dataframes(df_avo_nrb, df_fidas)

# Save combined dataframe
df_common.write_parquet(results / "nrb_avo_fidas_combined.parquet")


In [68]:
# Plot
plot_pm_comparison_from_dataframe(df_common, output_dir=results, width=7, height=6)

In [ ]:
# Nairobi AE31 data
instrument_name = "ae31"

# aggregate data
df_ae31, path_ae31 = aggregate_data(
    source=source,       # folder with .parquet files
    target=target,      # where output should go
    instrument_name=instrument_name,
    freq="hourly",
    statistics="mean"
)
df_ae31.write_parquet(path_ae31)

# plot timeseries

# Define channels and corresponding wavelengths (in nm)
channels = ["UV370", "B470", "G520", "Y590", "R660", "IR880", "IR950"]
wavelengths = [370, 470, 520, 590, 660, 880, 950]

# Generate colors
colors = _wavelength_to_color(wavelengths)

# Build dictionary
channels_and_colors = dict(zip(channels, colors))

plot_time_series(df_ae31, channels_and_colors, highlight_channel="IR880", output_path=results / "ae31_timeseries.png", figsize=(10, 5), plot_type="scatter")

# plot diurnal variability
plot_diurnal_by_wavelength(df_ae31, channels_and_colors, output_path=results / "ae31_diurnal_cycle_box.png", plot_type="box", figsize=(10, 5))

In [ ]:
df_avo_nrb.describe()

In [ ]:
# plot timeseries
channels_and_colors = {
    "Cn [1/cm³]": "#1f77b4",  # Blue       — for number concentration (distinct from mass)
    "PM1 [mg/m³]": "#2ca02c",  # Green      — PM1
    "PM2.5 [mg/m³]": "#ff7f0e",  # Orange     — PM2.5
    "PM4 [mg/m³]": "#d62728",  # Red        — PM4
    "PM10 [mg/m³]": "#9467bd",  # Purple     — PM10
    "PMtotal [mg/m³]": "#8c564b",  # Brown      — PMtotal
}

plot_time_series(df_avo_nrb, 
                 channels_and_colors=channels_and_colors, 
                 highlight_channel='Cn [1/cm³]', 
                 plot_type="line", 
                 output_path=results / "fidas_timeseries.png", 
                 figsize=(10, 5),
                 title="Fidas hourly data (Dagoretti Corner, NRB)",
                 ylabel=["Aerosol particle mass concentration [mg/m3]", "Aerosol particle number concentration [1/cm3]"],
                 which_y_axes=(2, 1, 1, 1, 1, 1),
                 legend_loc='upper left')
